In [ ]:
# Add repo root to Python path
import sys
from pathlib import Path

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"✅ Added to path: {repo_root}")

## Import modules

In [ ]:
from src.vectordb.pinecone_ltm import PineconeLTM
from src.embedding.embedder import HuggingFaceEmbedder, normalize_embeddings
from src.ingestion.loaders import DataLoader
from src.config import Config
import numpy as np

## Load chunks and embeddings

In [ ]:
# Load chunks
chunks_path = Config.MVP_KB_OUTPUTS / "mdn_chunks.jsonl"
chunks = DataLoader.load_jsonl(chunks_path)

# Load embeddings
embeddings_path = Config.MVP_KB_OUTPUTS / "chunk_embeddings.npy"
chunk_embeddings = np.load(embeddings_path)

print(f"📦 Loaded chunks: {len(chunks)}")
print(f"📊 Loaded embeddings: {chunk_embeddings.shape}")
print(f"✅ Data ready for Pinecone upload")

## Initialize Pinecone LTM

In [ ]:
# Initialize Pinecone (creates index if needed)
ltm = PineconeLTM(
    index_name="self-evolving-dce-ltm",
    embedding_dim=384,  # BGE-small dimension
    metric="cosine"
)

print(f"\n✅ Pinecone LTM initialized")
print(f"Index: {ltm.index_name}")
print(f"Dimension: {ltm.embedding_dim}")
print(f"Metric: {ltm.metric}")

## Check current index statistics

In [ ]:
# Get stats before upload
stats_before = ltm.get_stats()

print("📊 Index statistics (before):")
print(f"  Total vectors: {stats_before['total_vector_count']}")
print(f"  Dimension: {stats_before['dimension']}")
print(f"  Namespaces: {stats_before['namespaces']}")

## Upload chunks to Pinecone

⚠️ **Note:** This uploads all chunks to Pinecone. Run this only once unless you want to refresh the data.

In [ ]:
# Upload chunks with embeddings
num_upserted = ltm.upsert_chunks(
    chunks=chunks,
    embeddings=chunk_embeddings,
    batch_size=200
)

print(f"\n✅ Upload complete: {num_upserted} chunks")

## Check statistics after upload

In [ ]:
# Get updated stats
stats_after = ltm.get_stats()

print("📊 Index statistics (after):")
print(f"  Total vectors: {stats_after['total_vector_count']}")
print(f"  Namespaces: {stats_after['namespaces']}")

# Check kb_chunks namespace specifically
kb_stats = ltm.get_stats(namespace=ltm.NS_CHUNKS)
print(f"\n  KB chunks namespace: {kb_stats['vector_count']} vectors")

## Test semantic search with Pinecone

In [ ]:
# Initialize embedder for query
embedder = HuggingFaceEmbedder(model="BAAI/bge-small-en-v1.5")

# Create query
query = "Explain semantic HTML and accessibility"
query_vec = embedder.embed_query(query)

print(f"🔍 Query: '{query}'")
print(f"Query embedding shape: {query_vec.shape}")

In [ ]:
# Search Pinecone
results = ltm.query(
    query_vector=query_vec,
    top_k=5,
    namespace=ltm.NS_CHUNKS
)

print(f"\n✅ Found {len(results)} results\n")

for i, match in enumerate(results, 1):
    metadata = match.get('metadata', {})
    score = match.get('score', 0)
    
    print(f"{i}. Score: {score:.4f}")
    print(f"   ID: {match['id']}")
    print(f"   Title: {metadata.get('title', 'N/A')}")
    print(f"   Heading: {metadata.get('heading', 'N/A')}")
    print(f"   Doc ID: {metadata.get('doc_id', 'N/A')}")
    print(f"   Tokens: {metadata.get('tokens', 0)}")
    print()

## Test filtering by metadata

In [ ]:
# Search with metadata filter (e.g., only chunks with "accessibility" in title)
filtered_results = ltm.query(
    query_vector=query_vec,
    top_k=3,
    namespace=ltm.NS_CHUNKS,
    filter_dict={"title": {"$eq": "Accessibility"}}  # Exact match
)

print(f"🔍 Filtered search (Accessibility title):")
print(f"Found {len(filtered_results)} results\n")

for i, match in enumerate(filtered_results, 1):
    metadata = match.get('metadata', {})
    print(f"{i}. {metadata.get('title')} - {metadata.get('heading')} (score: {match.get('score', 0):.4f})")

---

## Summary

**What this demo does:**
1. Initializes Pinecone with proper index configuration
2. Uploads all chunks with embeddings to Pinecone
3. Demonstrates semantic search via vector similarity
4. Shows metadata filtering capabilities

**Benefits of Pinecone LTM:**
- ✅ Persistent storage (survives notebook restarts)
- ✅ Scalable vector search
- ✅ Multiple namespaces for different data types
- ✅ Metadata filtering
- ✅ Cross-session memory for agents

**Next step:** Build complete auto-RAG pipeline script